# Prática 1 — Feature Extraction em CIFAR-10
**ENG4502 — Introdução à Ciência de Dados · PUC-Rio**

Neste notebook você vai implementar, passo a passo, a técnica de **Feature Extraction** com Transfer Learning — a mesma estratégia demonstrada no notebook Colab de introdução, desta vez construída por você.

**O que é Feature Extraction?**  
A ResNet-18, treinada em 1,2 milhão de imagens do ImageNet, já aprendeu a reconhecer bordas, texturas, formas e partes de objetos. Em vez de treinar tudo isso novamente — o que exigiria muito dado e tempo —, **congelamos** esses pesos e treinamos apenas uma nova camada final de classificação adaptada ao CIFAR-10 (10 classes).

**Etapas deste notebook:**
1. Carregar o CIFAR-10 e preparar um subset balanceado.
2. Congelar todos os parâmetros convolucionais da ResNet-18.
3. Substituir a camada de classificação final por uma nova camada linear de 10 classes.
4. Treinar apenas essa nova camada usando SGD.
5. Avaliar a acurácia de validação e refletir sobre os resultados.

> Complete as seções marcadas com `### SEU CÓDIGO AQUI ###`.

## ▶️ Passo 0 — Ativar a GPU (importante!)

No menu do Colab: **Ambiente de execução → Alterar o tipo de ambiente de execução → Acelerador de hardware: GPU (T4)**.

Execute a célula abaixo para confirmar. Sem GPU, o modelo percorre a ResNet inteira a cada batch — cada época leva **~5–10 min na CPU**. Com GPU (T4), cai para menos de 1 min por época.

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'✅ GPU ativa: {torch.cuda.get_device_name(0)}')
else:
    device = torch.device('cpu')
    print('⚠️  GPU NÃO detectada — ative em: Ambiente de execução → Alterar o tipo → GPU (T4)')
    print('   Sem GPU, cada época leva ~5-10 min (5 épocas ≈ 30-50 min total).')

## 0. Imports e Configuração de Hardware

Carregamos as bibliotecas essenciais:
- **`torch` / `torch.nn` / `torch.optim`**: núcleo do PyTorch — tensores, camadas e otimizadores.
- **`torchvision`**: datasets prontos (CIFAR-10) e modelos pré-treinados (ResNet-18).
- **`Subset`**: seleciona um subconjunto do dataset lendo apenas os rótulos, sem carregar todas as imagens na memória.
- **`numpy` / `matplotlib`**: manipulação numérica e visualização.

A variável `device` (definida no Passo 0) diz ao PyTorch onde executar os cálculos — GPU (`cuda`) se disponível, CPU caso contrário. Ela será passada explicitamente para cada tensor e modelo.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import Subset
import numpy as np
import matplotlib.pyplot as plt
import time

# Reconfigura device caso esta célula seja executada antes do Passo 0
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando o dispositivo: {device}')

## 1. Carregamento dos Dados com Subset Balanceado

**Por que um subset?**  
O CIFAR-10 tem 50.000 imagens de treino. Processar tudo com a ResNet-18 em CPU seria lento demais para uma aula. Usamos **500 imagens por classe** (10 × 500 = **5.000 de treino**) e **100 por classe** para validação (**1.000 de validação**).

**Por que balanceado?**  
Se sortearmos aleatoriamente, algumas classes podem ter mais amostras do que outras, distorcendo treino e avaliação. A função `extract_balanced_subset` garante exatamente a mesma quantidade para cada uma das 10 classes — sem abrir uma única imagem, apenas lendo os rótulos (`dataset.targets`).

**Por que redimensionar para 224×224?**  
A ResNet-18 foi projetada para imagens de 224×224 pixels. As imagens do CIFAR-10 têm apenas 32×32 — precisamos ampliá-las para o tamanho esperado pelos filtros convolucionais pré-treinados.

**O que é a normalização ImageNet?**  
Os valores de pixel são reescalonados para ter a mesma média e desvio padrão usados no ImageNet (`mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]` por canal RGB). Isso é obrigatório porque os pesos pré-treinados foram aprendidos com dados nessa escala — usar outra degradaria as features extraídas.

**O que é um DataLoader?**  
`DataLoader` é um iterador que entrega lotes (*batches*) de imagens e rótulos ao modelo durante o treino. `shuffle=True` embaralha os dados a cada época. `num_workers=0` é necessário para compatibilidade com Windows e Colab.

In [ ]:
N_TRAIN = 5000
N_VAL = 1000

# Pipelines de transformação (ResNet-18 exige 224x224 e normalização do ImageNet)
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_full = datasets.CIFAR10(root='data', train=True, download=True, transform=transform)
val_full = datasets.CIFAR10(root='data', train=False, download=True, transform=transform)

def extract_balanced_subset(dataset, n_total):
    if n_total is None:
        return dataset
    n_per_class = n_total // 10
    indices = []
    class_counts = {c: 0 for c in range(10)}
    for idx, label in enumerate(dataset.targets):
        if class_counts[label] < n_per_class:
            indices.append(idx)
            class_counts[label] += 1
        if len(indices) == n_total:
            break
    return Subset(dataset, indices)

train_dataset = extract_balanced_subset(train_full, N_TRAIN)
val_dataset = extract_balanced_subset(val_full, N_VAL)

# ==========================================
# EXERCÍCIO 1: Crie os DataLoaders para train_dataset e val_dataset
# Dica: use batch_size=64, shuffle=True no treino e num_workers=0 (essencial para Colab e Windows)
# ==========================================
### SEU CÓDIGO AQUI ###
train_loader = None
val_loader = None

assert train_loader is not None, "❌ Exercício 1: crie o train_loader com DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)."
assert val_loader   is not None, "❌ Exercício 1: crie o val_loader com DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)."
print(f'Amostras de Treino: {len(train_dataset)} | Validação: {len(val_dataset)}')

## 2. Preparação do Modelo (Feature Extraction)

**O que é o backbone?**  
A ResNet-18 é formada por blocos convolucionais — o **backbone** — seguidos de uma camada de classificação linear (`model.fc`). O backbone extrai representações visuais ricas das imagens (um vetor de 512 valores por imagem); a `fc` mapeia essas representações para as classes do problema.

**Congelar parâmetros (`requires_grad = False`)**  
Ao setar `requires_grad = False`, dizemos ao PyTorch para **não calcular gradientes** para aqueles parâmetros durante o backpropagation — eles ficam "congelados" e seus valores não mudam durante o treino. Isso preserva o conhecimento visual aprendido no ImageNet intacto.

**Substituir a camada final (`model.fc`)**  
A ResNet-18 original termina com `nn.Linear(512, 1000)` (512 features do backbone → 1.000 classes do ImageNet). Trocamos por `nn.Linear(512, 10)` para as 10 classes do CIFAR-10. Esta nova camada começa com pesos aleatórios e será a **única** treinada.

**Validação esperada:** `trainable_params` deve ser exatamente **5.130** = 512 × 10 pesos + 10 bias da nova `fc`. O restante dos ~11 milhões de parâmetros permanece congelado.

In [ ]:
# ==========================================
# EXERCÍCIO 2: Configure a ResNet-18 para Feature Extraction
# 1. Carregue o modelo ResNet-18 pré-treinado com weights=models.ResNet18_Weights.IMAGENET1K_V1
# 2. Percorra model.parameters() congelando-os (requires_grad = False)
# 3. Substitua model.fc por uma nova nn.Linear adaptada para 10 classes
# ==========================================
### SEU CÓDIGO AQUI ###
model = None

assert model is not None, "❌ Exercício 2: defina o modelo (carregue a ResNet-18, congele os pesos e substitua model.fc por nn.Linear(512, 10))."

model = model.to(device)

# Contar parâmetros treináveis vs. congelados para validar o congelamento
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total de Parâmetros: {total_params:,}')
print(f'Parâmetros Treináveis: {trainable_params:,} (deve ser exatamente 5,130)')

## 3. Funções de Treinamento e Avaliação

**`train_epoch` — uma época de treino**  
Para cada batch: (1) move dados para o `device`; (2) zera gradientes acumulados (`zero_grad` — sem isso, gradientes se somam entre batches); (3) faz o *forward pass* e calcula os logits; (4) calcula a loss; (5) propaga gradientes para trás (`loss.backward()`); (6) atualiza os pesos (`optimizer.step()`). `running_loss` acumula a perda ponderada pelo tamanho de cada batch para calcular a média correta ao final da época.

**`evaluate` — acurácia de validação**  
`model.eval()` desativa dropout e usa estatísticas fixas de batch norm. `torch.no_grad()` desliga o cálculo de gradientes — desnecessário na avaliação, economiza memória e acelera o processo. `torch.max(outputs, 1)` retorna o índice da classe com maior logit (a predição do modelo).

**`criterion` — `CrossEntropyLoss`**  
Função de perda padrão para classificação multi-classe. Combina `LogSoftmax` + `NLLLoss` internamente, o que a torna numericamente estável.

**`optimizer` — SGD com momentum**  
Passamos `model.fc.parameters()` — não `model.parameters()` — para garantir que **somente** os pesos da camada final sejam atualizados. `lr=0.01` é adequado para treinar apenas a nova camada linear. `momentum=0.9` acelera a convergência em direções consistentes e amortece oscilações.

In [ ]:
# ==========================================
# EXERCÍCIO 3: Defina a Loss e o Otimizador
# Dica: Garanta que o SGD receba para otimização APENAS os parâmetros da fc (model.fc.parameters())
# Use learning rate lr=0.01 e momentum=0.9
# ==========================================
### SEU CÓDIGO AQUI ###
criterion = None
optimizer = None

assert criterion is not None, "❌ Exercício 3: defina criterion (ex: nn.CrossEntropyLoss())."
assert optimizer  is not None, "❌ Exercício 3: defina optimizer (ex: optim.SGD(model.fc.parameters(), lr=0.01, momentum=0.9))."

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    return running_loss / len(loader.dataset)

def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

## 4. Execução do Experimento

Rodamos por **5 épocas**. O que observar:
- **Acurácia desde a 1ª época:** espera-se começar em torno de 60–70% — esse salto inicial é o poder do Transfer Learning. O backbone já sabe extrair features visuais relevantes; basta treinar o classificador final.
- **Loss decrescente:** deve cair progressivamente a cada época, indicando que o classificador está convergindo.
- **Tempo por época:** com GPU < 1 min; com CPU ~5–10 min.

**Benchmark esperado:** ~70–75% de acurácia de validação após 5 épocas. Para comparação, treinar a mesma rede do zero com apenas 5.000 exemplos daria ~38%.

In [ ]:
num_epochs = 5
print('=== Iniciando Treinamento (Feature Extraction) ===')
t_start = time.time()

for epoch in range(1, num_epochs + 1):
    t0 = time.time()
    loss = train_epoch(model, train_loader, criterion, optimizer, device)
    acc = evaluate(model, val_loader, device)
    elapsed = time.time() - t0
    print(f'Época {epoch}/{num_epochs} | Loss: {loss:.4f} | Val Acc: {acc*100:.2f}% | Tempo: {elapsed:.1f}s')

total_time = time.time() - t_start
print(f'Treinamento completo em {total_time/60:.1f} minutos.')

## 5. Questões de Reflexão

Responda com base nos resultados que você observou:

1. **Por que a acurácia de validação já começa alta desde a primeira época**, em comparação a treinar uma rede do zero? Pense no papel dos pesos congelados do backbone — o que eles já "sabem" fazer?

2. **O que aconteceria** se tivéssemos passado `model.parameters()` (todos os parâmetros) ao otimizador SGD em vez de `model.fc.parameters()`? Os pesos do backbone seriam atualizados mesmo estando com `requires_grad=False`? Por quê ou por que não?